# Stage 4：Olist 訂單是否延遲預測

這本 Notebook 帶你完成「訂單核准時，預測最後是否超過預估送達日期」的流程。特徵與訓練邏輯由 `scripts/delay_features.py`、`scripts/train_delay_model.py` 共用，這裡負責導覽、檢查結果與示範推論。

請使用 **Shop (Python 3.12)** kernel。資料來自本機 `data/raw/olist/` 的六份 CSV，執行時不需要 MySQL。模型與大量逐筆預測保留在本機，Notebook 不內嵌這些檔案。


## 1. 先定義預測時點與答案

- **預測時點**：`order_approved_at`，也就是訂單／付款核准時間。
- **延遲標記**：實際送達的「日期」晚於預估送達的「日期」時，`delayed = 1`，其餘為 0。
- **答案可得時間**：`label_available_time` 是實際送達時間。訓練時必須檢查當時是否已知道答案。

例如預估 1 月 10 日送達，實際 1 月 10 日 23:59 送達仍算準時；1 月 11 日 00:00 送達才算延遲。Stage 3 用完整時間戳記比較午夜的統計方式與這裡不同，因此延遲率可能不同。

歷史訓練資料限於已送達、所需時間完整且時序合理的訂單。這個模型預測的是這群訂單的配送延遲，不包含取消或始終未送達的風險。


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import joblib
import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, cwd.parent)
     if (candidate / "scripts" / "delay_features.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("請從專案根目錄或 notebooks 資料夾執行這本 Notebook。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.delay_features import (
    CATEGORICAL_FEATURES, FEATURES, NUMERIC_FEATURES, build_delay_dataset,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
MODEL_PATH = PROJECT_ROOT / "models" / "olist_delay_model.joblib"
METRICS_PATH = PROJECT_ROOT / "reports" / "stage4" / "metrics.json"

print("專案根目錄：", PROJECT_ROOT)
print("Python：", sys.executable)


## 2. 建立每訂單一列的資料

商品明細與付款紀錄會各自彙總，再合併到訂單，避免一張訂單有多個商品與多筆付款時把金額重複加總。下面只顯示資料量、延遲比例與特徵摘要。


In [ ]:
dataset = build_delay_dataset(RAW_DIR)

display(pd.DataFrame([{
    "合格訂單數": dataset.shape[0],
    "資料欄位數": dataset.shape[1],
    "模型特徵數": len(FEATURES),
    "延遲訂單數": int(dataset["delayed"].sum()),
    "延遲比例": float(dataset["delayed"].mean()),
    "最早核准時間": dataset["prediction_time"].min(),
    "最晚核准時間": dataset["prediction_time"].max(),
}]))


## 3. 核准時可以知道哪些特徵

數值特徵包含商品數、商品與運費總額、付款資訊、商品重量與體積、預估配送天數、核准等待時間、購買時段及跨州商品比例。類別特徵包含顧客州別、主要賣家州別、主要品類與付款方式。完整欄位使用明確白名單 `FEATURES`。

實際送達、交給物流的時間、評論、最終訂單狀態，以及訂單／顧客 ID 都不進入模型。前幾項屬於預測之後才知道的資訊，ID 也不作為預測特徵。缺值補齊、標準化與類別編碼都放在 Pipeline，只用訓練集學習。

Olist 是事後快照資料，沒有每次商品、付款或預估交期變更的完整時間紀錄。本階段假設這些欄位在核准時已知；未來接真實系統時，應保存核准當下的快照來驗證這個假設。


In [ ]:
feature_summary = pd.DataFrame({
    "feature": FEATURES,
    "kind": ["numeric" if name in NUMERIC_FEATURES else "categorical"
             for name in FEATURES],
    "missing_rate": [float(dataset[name].isna().mean()) for name in FEATURES],
    "distinct_values": [dataset[name].nunique(dropna=True) for name in FEATURES],
})
display(feature_summary)


## 4. 依時間切分，排除當時還不知道的答案

訓練程式依 `prediction_time` 排序，以約 60%／20%／20% 的訂單量建立 train、validation、test；同一天的訂單全部放在同一區間，邊界日歸到後面的區間。

只做時間切分仍可能用到未來答案，因此還會做 **label maturity purge**：

- train 的送達時間必須嚴格早於 validation 開始時間。
- validation 的送達時間必須嚴格早於 test 開始時間。

因此三組實際筆數會受到日期邊界與 purge 影響。這次保留選定的原始 train 模型，不在選模後合併 validation 重訓；模型、前處理與門檻先固定，再對 test 預測一次。


## 5. 使用現有訓練成果，必要時由 CLI 建立

`RUN_TRAINING = False` 時，模型與報告都存在就直接讀取；任一檔案不存在時會自動執行訓練。若要重新產生整套成果，把設定改為 `True`。

CLI 使用目前 kernel 的 `sys.executable`，確保與 Notebook 使用同一個 Python 環境。


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING or not MODEL_PATH.is_file() or not METRICS_PATH.is_file():
    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "train_delay_model.py"),
        "--data-dir", str(RAW_DIR),
        "--output-dir", str(METRICS_PATH.parent),
        "--model-path", str(MODEL_PATH),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("使用已存在的模型與 metrics.json。")


In [ ]:
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
artifact = joblib.load(MODEL_PATH)

if artifact["feature_names"] != list(FEATURES):
    raise ValueError("模型特徵與目前程式不同，請重新訓練。")
if artifact["selected_model"] != metrics["selected_model"]:
    raise ValueError("模型與報告不一致，請重新產生同一批訓練成果。")

split_info = metrics["metadata"]["split"]
print("Validation 開始：", split_info["validation_start"])
print("Test 開始：", split_info["test_start"])
display(pd.DataFrame.from_dict(split_info["partitions"], orient="index"))


## 6. 在 validation 選模型與決策門檻

這次比較 Dummy baseline、Logistic Regression、Random Forest。Dummy 只作為基本參考；可選模型是 Logistic Regression 與 Random Forest，分別學習線性關係與樹模型的非線性關係。

先在兩個可選模型中，以 validation 的 **Average Precision（AP）最高者選模型**，並檢查 AP 是否高於 Dummy。AP 衡量排序能力，在延遲訂單較少的資料中特別有參考價值。再只使用被選模型的 validation 分數，選擇 **F1 最高的門檻**。F1 同時考慮 precision 與 recall；這是本階段的實驗策略，未來可依人工查核量或延遲漏判成本調整。

下表比較候選模型的 validation 結果。這些分類指標使用 0.5 門檻；AP 和 ROC AUC 使用連續分數，不受分類門檻影響。


In [ ]:
validation_rows = []
for model_name, result in metrics["model_comparison"].items():
    row = {"model": model_name, "status": result["status"]}
    if result["status"] == "ok":
        row.update(result["validation_at_0_5"])
        row["fit_seconds"] = result["fit_seconds"]
    else:
        row["error"] = result.get("error", "")
    validation_rows.append(row)

models_validation = pd.DataFrame(validation_rows)
display(models_validation)


In [ ]:
selected = pd.DataFrame([{
    "selected_model": artifact["selected_model"],
    "threshold": float(artifact["threshold"]),
    "selection_metric": "validation average_precision",
    "threshold_objective": "validation F1",
    "validation_AP": metrics["validation"]["average_precision"],
    "validation_AP_above_dummy": metrics["metadata"]["model_selection"]["validation_ap_above_dummy"],
    "validation_F1_at_selected_threshold": metrics["validation"]["f1"],
}])
display(selected)


## 7. 最後才看 test，記錄保留資料的表現

Test 不參與補值、編碼、模型選擇或門檻選擇。下面讀取凍結 Pipeline 在 test 的一次評估結果，並與全猜準時的固定 baseline 比較，不使用 test 結果反覆調參。

- **Precision**：判成延遲的訂單中，實際延遲的比例。
- **Recall**：實際延遲的訂單中，被模型抓到的比例。
- **F1**：precision 與 recall 的調和平均。
- **AP／ROC AUC**：衡量連續分數的排序能力。
- **Accuracy／balanced accuracy**：作為補充，需一起考慮延遲與準時的比例。


In [ ]:
test_keys = [
    "rows", "positive_rate", "threshold", "average_precision", "roc_auc",
    "f1", "recall", "precision", "balanced_accuracy", "accuracy",
]
testmetrics = pd.DataFrame([
    {"model": artifact["selected_model"],
     **{key: metrics["test"][key] for key in test_keys}},
    {"model": "all_on_time_baseline",
     **{key: metrics["test_baseline"][key] for key in test_keys}},
])
display(testmetrics)

confusion = pd.DataFrame(
    metrics["test"]["confusion_matrix"],
    index=["實際準時", "實際延遲"],
    columns=["預測準時", "預測延遲"],
)
display(confusion)
display(Image(filename=str(
    PROJECT_ROOT / "reports" / "images" / "stage4_test_confusion_matrix.png"
)))


## 8. 看哪些特徵影響 validation 的排序

Permutation importance 會打亂某一欄，再看 validation AP 下降多少。下降越多，代表被選模型較依賴該欄；負值或接近零代表這次抽樣沒有顯示穩定的正面貢獻。

圖表使用 validation 抽樣，未用 test 來挑特徵。這是模型行為的描述，不能當成延遲的因果證據；高度相關的欄位也可能分攤重要性。


In [ ]:
display(Image(filename=str(
    PROJECT_ROOT / "reports" / "images" / "stage4_validation_permutation_importance.png"
)))


## 9. 用一筆資料示範 Pipeline 推論

這裡取歷史資料的一筆來示範介面，只把白名單特徵交給儲存的 Pipeline。前處理會沿用訓練時學到的設定；決策門檻沿用 artifact 內 validation 選出的值。

`predict_proba` 輸出作為延遲分數使用。這次沒有做機率校準，分數不應直接解讀成已校準的真實延遲機率。這一筆示範也不構成額外的效能評估。


In [ ]:
example = dataset.iloc[[0]]
feature_row = example.loc[:, artifact["feature_names"]].copy()
pipeline = artifact["pipeline"]
positive_class_index = list(pipeline.classes_).index(1)
delay_score = float(pipeline.predict_proba(feature_row)[0, positive_class_index])
threshold = float(artifact["threshold"])

display(pd.DataFrame([{
    "order_id": example["order_id"].iloc[0],
    "model": artifact["selected_model"],
    "delay_score_uncalibrated": delay_score,
    "threshold": threshold,
    "predicted_delayed": int(delay_score >= threshold),
}]))


## 本階段留下的成果

- 可重用的訂單特徵與延遲標記建置程式。
- 使用時間切分與成熟標記 purge 的模型比較。
- 固定模型、前處理與決策門檻的本機 artifact。
- Validation 選模結果、test 指標、混淆矩陣與特徵重要性圖。
- 可用同一套 Pipeline 進行推論的介面。

下一步可依專案目標選擇：把這個模型接成 API、討論延遲判斷的商業成本與門檻，或繼續 Stage 5 的評論好壞預測。這些工作需要各自的資料與評估設計。
